# Closed-Loop LQR Simulation

Runs the full nonlinear plant with the LQR controller in the loop. Produces diagnostic plots showing position, velocity, acceleration, attitude, angular rate, and control inputs over time.

**Mission:** start at ground level tilted 5 degrees, climb to 4 m altitude, hover.

Modify the initial conditions, target, and simulation duration below to test different scenarios.

## Imports and setup

Note: your controller module is named `control.py`, so we import from `control` (not `controller`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Import from your modules
from dynamics import nonlinear_dynamics, mass, gravity
from lander_control import LQRController, saturate

# Make plots render inline and crisper
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 4)

# Quick sanity check on sign conventions
print(f'mass = {mass} kg')
print(f'gravity = {gravity} m/s^2 (negative because it points in -Z)')
print(f'trim thrust = -mass*gravity = {-mass*gravity:.3f} N')

## Controller design

Uses the Q and R weights from your `control.py`. Edit here to re-tune without modifying the source file.

In [ ]:
Q = np.diag([
    10, 10, 10,     # position (x, y, z)
    10, 10, 10,     # velocity (vx, vy, vz)
    1, 1, 1,        # attitude (theta_x, theta_y, theta_z)
    0.5, 0.5, 0.5   # angular rate (omega_x, omega_y, omega_z)
])

R = np.diag([
    50,    # thrust
    20,      # gimbal x
    20,      # gimbal y
    50      # yaw torque
])

controller = LQRController(Q, R)
print('Controller designed. K shape:', controller.K.shape)
print(f'u_trim = {controller.u_trim}')

## Simulation setup

Starting condition: vehicle on ground, tilted 5 degrees about body X.  
Target: hover at 4 m altitude.

In [ ]:
# Target state: hover at 4m altitude, everything else zero
x_target = np.zeros(12)
x_target[2] = 4.0   # target z (altitude) in meters

# Initial state: on the ground, tilted 5 degrees about body X
x0 = np.zeros(12)
x0[6] = np.radians(5)
x0[5] = np.radians(5)

# Simulation duration (seconds)
t_end = 30.0

# Closed-loop dynamics: controller output feeds plant (with actuation limits)
def closed_loop_dynamics(t, x):
    u = saturate(controller.compute(x, x_target))
    return nonlinear_dynamics(x, u)

# Run the simulation
sol = solve_ivp(
    closed_loop_dynamics,
    t_span=[0, t_end],
    y0=x0,
    max_step=0.01,
    dense_output=True
)

# Sample at uniform times for clean plotting
t = np.linspace(0, t_end, 1000)
x_traj = sol.sol(t)   # shape: (12, 1000)

# Reconstruct control inputs at each timestep (with saturation applied)
u_traj = np.array([saturate(controller.compute(x_traj[:, i], x_target)) for i in range(len(t))]).T

# Reconstruct accelerations at each timestep (from nonlinear dynamics output)
xdot_traj = np.array([nonlinear_dynamics(x_traj[:, i], u_traj[:, i]) for i in range(len(t))]).T

print(f'Simulation complete: {len(t)} samples over {t_end} seconds')

## Position plot

Expect Z to rise smoothly from 0 to 4 m. X and Y should stay near zero, with small excursions during tilt recovery.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x_traj[0], label='x', linewidth=2)
ax.plot(t, x_traj[1], label='y', linewidth=2)
ax.plot(t, x_traj[2], label='z', linewidth=2)
ax.axhline(x_target[2], color='k', linestyle='--', alpha=0.5, label='z target')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Position (m)')
ax.set_title('Position over time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Velocity plot

Expect Z velocity to ramp up during climb, peak, then decay to zero at altitude. X and Y velocities should be small transients during tilt correction.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x_traj[3], label='vx', linewidth=2)
ax.plot(t, x_traj[4], label='vy', linewidth=2)
ax.plot(t, x_traj[5], label='vz', linewidth=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Velocity (m/s)')
ax.set_title('Velocity over time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Acceleration plot

These come directly from `nonlinear_dynamics` output (xdot values in positions 3-5). Shows the actual physical accelerations the vehicle experiences. Expect sharp initial transients during tilt recovery and takeoff, then smooth decay.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, xdot_traj[3], label='ax', linewidth=2)
ax.plot(t, xdot_traj[4], label='ay', linewidth=2)
ax.plot(t, xdot_traj[5], label='az', linewidth=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Acceleration (m/s^2)')
ax.set_title('Acceleration over time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Attitude plot

Expect theta_x to start at 5 degrees, drop to near zero within ~1 second. Other angles stay near zero.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, np.degrees(x_traj[6]), label='theta_x', linewidth=2)
ax.plot(t, np.degrees(x_traj[7]), label='theta_y', linewidth=2)
ax.plot(t, np.degrees(x_traj[8]), label='theta_z', linewidth=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Attitude (deg)')
ax.set_title('Attitude over time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Angular rate plot

These are what the gyroscope would measure. Expect a sharp initial transient as the controller corrects the tilt, then settling to zero.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, np.degrees(x_traj[9]), label='omega_x', linewidth=2)
ax.plot(t, np.degrees(x_traj[10]), label='omega_y', linewidth=2)
ax.plot(t, np.degrees(x_traj[11]), label='omega_z', linewidth=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angular rate (deg/s)')
ax.set_title('Angular rate over time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Control inputs plot

What the controller is commanding. Watch for:
- Thrust should stay near trim (`-mass*gravity = 8.63 N`). Large transients during takeoff are fine.
- Gimbal angles should stay well within +/- 10 degrees. If they saturate, raise R on gimbal channels.
- Yaw torque should be small (your current R[3,3] = 50 keeps it reasonable).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Thrust (separate subplot because units are different)
axes[0].plot(t, u_traj[0], label='Thrust (N)', linewidth=2, color='tab:blue')
axes[0].axhline(-mass*gravity, color='k', linestyle='--', alpha=0.5, label='Trim thrust')
axes[0].set_ylabel('Thrust (N)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Control inputs over time')

# Gimbal angles and yaw torque
axes[1].plot(t, np.degrees(u_traj[1]), label='Gimbal X (deg)', linewidth=2, color='tab:orange')
axes[1].plot(t, np.degrees(u_traj[2]), label='Gimbal Y (deg)', linewidth=2, color='tab:green')
axes[1].plot(t, u_traj[3], label='Yaw torque (N*m)', linewidth=2, color='tab:red')
axes[1].axhline(10, color='r', linestyle=':', alpha=0.5, label='Gimbal saturation +/-10 deg')
axes[1].axhline(-10, color='r', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Gimbal angle (deg) / Yaw torque (N*m)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Performance metrics

Summary numbers to evaluate controller performance.

In [ ]:
# Altitude tracking
z_final = x_traj[2, -1]
z_error = z_final - x_target[2]
z_max = np.max(x_traj[2])
z_overshoot_pct = 100 * (z_max - x_target[2]) / x_target[2] if z_max > x_target[2] else 0

# Attitude recovery (first time |theta_x| drops below 0.5 deg)
theta_x_deg = np.degrees(x_traj[6])
below_threshold = np.abs(theta_x_deg) < 0.5
if np.any(below_threshold):
    theta_x_settle_time = t[np.argmax(below_threshold)]
else:
    theta_x_settle_time = None

# Lateral drift during tilt recovery
max_lateral_drift = np.max(np.sqrt(x_traj[0]**2 + x_traj[1]**2))

# Gimbal saturation check
max_gimbal_x = np.max(np.abs(np.degrees(u_traj[1])))
max_gimbal_y = np.max(np.abs(np.degrees(u_traj[2])))

# Peak thrust
max_thrust = np.max(u_traj[0])
min_thrust = np.min(u_traj[0])

print('=' * 55)
print('Closed-loop simulation metrics')
print('=' * 55)
print(f'Final altitude:          {z_final:.3f} m (target: {x_target[2]:.1f} m)')
print(f'Altitude error:          {z_error*1000:.1f} mm')
print(f'Altitude overshoot:      {z_overshoot_pct:.2f} %')
if theta_x_settle_time is not None:
    print(f'Tilt settling time:      {theta_x_settle_time:.2f} s (to < 0.5 deg)')
else:
    print(f'Tilt settling time:      not reached in {t_end} s')
print(f'Max lateral drift:       {max_lateral_drift*1000:.1f} mm')
print(f'Max gimbal X command:    {max_gimbal_x:.2f} deg  (limit: 10 deg)')
print(f'Max gimbal Y command:    {max_gimbal_y:.2f} deg  (limit: 10 deg)')
print(f'Peak thrust:             {max_thrust:.3f} N')
print(f'Min thrust:              {min_thrust:.3f} N')
print('=' * 55)

## What to look for

**Good signs:**
- Altitude settles at 4.0 m within a few seconds with little overshoot
- Tilt returns to near zero within ~1 second
- Gimbal commands stay within +/- 10 degrees
- Lateral drift is small (< 100 mm)
- All traces converge smoothly to target

**Warning signs:**
- Gimbal commands saturate at +/- 10 degrees → raise R on gimbal channels
- Lots of oscillation → raise Q on velocity states (adds damping)
- Slow convergence → raise Q on position states
- Yaw torque spikes very high → raise R on yaw channel
- Vehicle falls or runs away → sign error somewhere in dynamics or controller

## Try these variations

Once the nominal case works, test edge cases:

```python
# Larger initial tilt (tests recovery from bigger disturbance)
x0[6] = np.radians(15)

# Initial angular rate (tests damping of rotation)
x0[9] = 1.0

# Lateral setpoint change (tests position tracking)
x_target[0] = 2.0

# Yaw setpoint change (tests yaw response)
x_target[8] = np.radians(30)
```